In [1]:
!pip install monai nibabel


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 44.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 103.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 79.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 13.7 MB/s eta 0:00:00:00:0100:01
  Attempting uninst

In [2]:
import os
import glob
import numpy as np
import torch
from monai.transforms import (
    LoadImaged, EnsureChannelFirstd, Spacingd, Orientationd,
    ScaleIntensityd, ResizeWithPadOrCropd, EnsureTyped,
    Compose, ToTensord
)
from monai.data import Dataset, DataLoader
from monai.networks.nets import resnet
from monai.networks.nets import resnet50
from monai.apps import download_and_extract
from monai.metrics import ConfusionMatrixMetric
import torch.nn as nn
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from monai.networks.nets import resnet

<frozen importlib._bootstrap_external>:1241: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
2025-07-20 20:11:03.417285: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753042263.568475      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753042263.612318      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
data_dir = "/kaggle/input/dataset/IXI-T1-sorted"
classes = sorted(os.listdir(data_dir))
class_map = {cls_name: idx for idx, cls_name in enumerate(classes)}

data_dicts = []
for cls_name in classes:
    full_folder = os.path.join(data_dir, cls_name)
    nii_files = glob.glob(os.path.join(full_folder, "*/*.nii"))
    for f in nii_files:
        data_dicts.append({"image": f, "label": class_map[cls_name]})

#print(data_dicts)


In [5]:
transforms = Compose([
    LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    Spacingd(keys=["image"], pixdim=(1.5, 1.5, 1.5), mode="bilinear"),
    Orientationd(keys=["image"], axcodes="RAS"),
    ScaleIntensityd(keys=["image"]),
    ResizeWithPadOrCropd(keys=["image"], spatial_size=(96, 96, 96)),
    EnsureTyped(keys=["image", "label"]),
    ToTensord(keys=["image", "label"])
])


In [6]:
train_val_files, test_files = train_test_split(
    data_dicts, 
    test_size=0.2, 
    stratify=[d['label'] for d in data_dicts],
    random_state=42
)


train_files, val_files = train_test_split(
    train_val_files, 
    test_size=0.2, 
    stratify=[d['label'] for d in train_val_files],
    random_state=42
)

In [7]:
train_ds = Dataset(data=train_files, transform=transforms)
val_ds = Dataset(data=val_files, transform=transforms)
test_ds = Dataset(data=test_files, transform=transforms)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=4)
val_loader = DataLoader(val_ds, batch_size=2, num_workers=4)
test_loader = DataLoader(test_ds, batch_size=2, num_workers=4)


In [8]:
from monai.networks.nets import resnet50
import torch
from collections import OrderedDict


model = resnet50(
    pretrained=False,
    n_input_channels=1,
    num_classes=5
)


checkpoint = torch.load('/kaggle/input/resnet_50_23/pytorch/default/1/resnet_50_23dataset.pth', map_location='cpu')


print("Checkpoint keys:", checkpoint.keys())  


state_dict = checkpoint['state_dict']


cleaned_state_dict = OrderedDict()
for k, v in state_dict.items():
    new_k = k.replace("module.", "").replace("model.", "").replace("backbone.", "")
    cleaned_state_dict[new_k] = v


missing, unexpected = model.load_state_dict(cleaned_state_dict, strict=False)


print("Missing keys:", missing)
print("Unexpected keys:", unexpected)


Checkpoint keys: dict_keys(['state_dict'])
Missing keys: ['layer1.0.downsample.0.bias', 'layer2.0.downsample.0.bias', 'layer3.0.downsample.0.bias', 'layer4.0.downsample.0.bias', 'fc.weight', 'fc.bias']
Unexpected keys: []


In [9]:
criterion = nn.CrossEntropyLoss()  
optimizer = torch.optim.Adam(model.parameters(), lr= 1e-4)  
accuracy_metric = ConfusionMatrixMetric(metric_name="accuracy")  
model.to(device)

ResNet(
  (conv1): Conv3d(1, 64, kernel_size=(7, 7, 7), stride=(1, 1, 1), padding=(3, 3, 3), bias=False)
  (bn1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (act): ReLU(inplace=True)
  (maxpool): MaxPool3d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): ResNetBottleneck(
      (conv1): Conv3d(64, 64, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
      (bn1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv3d(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
      (bn2): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv3d(64, 256, kernel_size=(1, 1, 1), stride=(1, 1, 1), bias=False)
      (bn3): BatchNorm3d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (act): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv3d(64, 25

In [10]:
def accuracy_fn(outputs, labels):
    preds = torch.argmax(outputs, dim=1)
    return (preds == labels).float().mean().item()


best_val_acc = 0.0
best_model_path = "resnet50_3d_best.pth"
num_epochs = 80

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_acc = 0.0

    loop = tqdm(train_loader, desc=f"[TRAIN] Epoch {epoch+1}/{num_epochs}", leave=False)
    for batch in loop:
        inputs, labels = batch["image"].to(device), batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        acc = accuracy_fn(outputs, labels)
        train_loss += loss.item()
        train_acc += acc

        loop.set_postfix(loss=loss.item(), acc=acc)

    avg_train_loss = train_loss / len(train_loader)
    avg_train_acc = train_acc / len(train_loader)


    model.eval()
    val_loss = 0.0
    val_acc = 0.0

    with torch.no_grad():
        val_loop = tqdm(test_loader, desc=f"[VAL] Epoch {epoch+1}/{num_epochs}", leave=False)
        for batch in val_loop:
            inputs, labels = batch["image"].to(device), batch["label"].to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            acc = accuracy_fn(outputs, labels)

            val_loss += loss.item()
            val_acc += acc
            val_loop.set_postfix(loss=loss.item(), acc=acc)

    avg_val_loss = val_loss / len(test_loader)
    avg_val_acc = val_acc / len(test_loader)

    print(f"\nEpoch [{epoch+1}/{num_epochs}]  Train Acc: {avg_train_acc:.4f} | Val Acc: {avg_val_acc:.4f}")

   
    if avg_val_acc > best_val_acc:
        best_val_acc = avg_val_acc
        torch.save(model.state_dict(), best_model_path)
        print(f" New best model saved at epoch {epoch+1} with Val Acc: {best_val_acc:.4f}")



Epoch [1/80]  Train Acc: 0.4146 | Val Acc: 0.4615
 New best model saved at epoch 1 with Val Acc: 0.4615



Epoch [2/80]  Train Acc: 0.4024 | Val Acc: 0.2692



Epoch [3/80]  Train Acc: 0.3841 | Val Acc: 0.2308



Epoch [4/80]  Train Acc: 0.3841 | Val Acc: 0.3077



Epoch [5/80]  Train Acc: 0.3780 | Val Acc: 0.4231



Epoch [6/80]  Train Acc: 0.4451 | Val Acc: 0.3846


... (70 outputs left out)


Epoch [77/80]  Train Acc: 0.9939 | Val Acc: 0.5962



Epoch [78/80]  Train Acc: 1.0000 | Val Acc: 0.5962



Epoch [79/80]  Train Acc: 0.9939 | Val Acc: 0.5192



Epoch [80/80]  Train Acc: 1.0000 | Val Acc: 0.5769


In [11]:

model.load_state_dict(torch.load(best_model_path))
model.eval()

test_correct = 0
test_total = 0

with torch.no_grad():
    for batch in tqdm(test_loader, desc="[TEST]"):
        inputs, labels = batch["image"].to(device), batch["label"].to(device)
        outputs = model(inputs)
        preds = torch.argmax(outputs, dim=1)
        test_correct += (preds == labels).sum().item()
        test_total += labels.size(0)

test_acc = test_correct / test_total
print(f"\nFinal Test Accuracy: {test_acc:.4f}")


[TEST]: 100%|██████████| 26/26 [00:11<00:00,  2.20it/s]


Final Test Accuracy: 0.6863
